In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from utils import load_data, evaluate_predictions, TARGET_COLUMNS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Vocabulary & Tokenizer
class SimpleVocab:
    def __init__(self, max_vocab=15000):
        self.max_vocab = max_vocab
        self.w2i = {"<PAD>": 0, "<UNK>": 1}

    def fit(self, texts):
        counts = {}
        for t in texts:
            for w in t.split():
                counts[w] = counts.get(w, 0) + 1
        top_words = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:self.max_vocab - 2]
        for idx, (w, _) in enumerate(top_words):
            self.w2i[w] = idx + 2

    def transform(self, text, max_len=150):
        tokens = [self.w2i.get(w, 1) for w in text.split()][:max_len]
        if len(tokens) < max_len:
            tokens += [0] * (max_len - len(tokens))
        return tokens

# 2. Dataset
class ToxicLSTMDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=150):
        self.X = [vocab.transform(t, max_len) for t in texts]
        self.y = labels

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.float32)
        )

# 3. Model Architecture
class MultiLabelBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=64, num_labels=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, x):
        emb = self.embedding(x)
        out, (hn, _) = self.lstm(emb)
        # Concatenate forward and backward final hidden states
        hidden = torch.cat((hn[-2, :, :], hn[-1, :, :]), dim=1)
        hidden = self.dropout(hidden)
        return self.fc(hidden)

# 4. Main Training Pipeline
train_df, val_df = load_data("train.csv")
vocab = SimpleVocab()
vocab.fit(train_df["cleaned_text"].values)

train_ds = ToxicLSTMDataset(train_df["cleaned_text"].values, train_df[TARGET_COLUMNS].values, vocab)
val_ds = ToxicLSTMDataset(val_df["cleaned_text"].values, val_df[TARGET_COLUMNS].values, vocab)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

model = MultiLabelBiLSTM(vocab_size=len(vocab.w2i), num_labels=len(TARGET_COLUMNS)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training Bi-LSTM...")
model.train()
for epoch in range(4):
    total_loss = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/4 - Loss: {total_loss / len(train_loader):.4f}")

# Evaluation
model.eval()
all_probs = []
with torch.no_grad():
    for bx, _ in val_loader:
        bx = bx.to(device)
        logits = model(bx)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)

y_probs = np.vstack(all_probs)
evaluate_predictions("Multi-Label Bi-LSTM", val_df[TARGET_COLUMNS].values.astype(np.float32), y_probs)